# Class 1 - Prompt Structure & Best Practices

**Week 4: Prompt Engineering (Basic to Intermediate)**

### Learning objectives
By the end of this notebook you will be able to:
- Name the five ingredients of a well-structured prompt.
- Rewrite a vague prompt into a specific, testable one.
- Explain the difference between a system prompt and a user prompt, and show how each shapes the output.
- Use delimiters and instruction ordering to make long prompts easier for a model to follow.
- Compare three drafts of the same prompt and explain why the final version is more reliable.


## Setup

Run this cell first. It reads your API key from the environment - never hardcode a key in a notebook.

Set one of these in your shell before launching Jupyter:

```bash
export OPENAI_API_KEY="sk-..."
# or
export ANTHROPIC_API_KEY="sk-ant-..."
```

You'll also need the matching SDK installed:

```bash
pip install openai anthropic
```


In [ ]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not OPENAI_API_KEY and not ANTHROPIC_API_KEY:
    print(
        "No API key found in the environment.\n"
        "Set one of these before running the demo cells below:\n"
        "  export OPENAI_API_KEY='sk-...'\n"
        "  export ANTHROPIC_API_KEY='sk-ant-...'\n"
        "The code below will run as soon as a key is set - nothing is hardcoded here."
    )
else:
    provider = "OpenAI" if OPENAI_API_KEY else "Anthropic"
    print(f"Found an API key for {provider}. You're ready to run the demo cells below.")

# --- Minimal provider-agnostic chat helper -------------------------------
# Uses OpenAI if OPENAI_API_KEY is set, otherwise falls back to Anthropic.

def ask(system_prompt, user_prompt, model=None, temperature=0.7, max_tokens=400):
    """Send a system + user prompt to whichever provider has a key set.
    Returns the assistant's text reply as a plain string."""
    if OPENAI_API_KEY:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        model = model or "gpt-4o-mini"
        resp = client.chat.completions.create(
            model=model,
            temperature=temperature,
            max_tokens=max_tokens,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        return resp.choices[0].message.content
    elif ANTHROPIC_API_KEY:
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        model = model or "claude-3-5-haiku-20241022"
        resp = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": user_prompt}],
        )
        return resp.content[0].text
    else:
        raise RuntimeError(
            "No API key set. Set OPENAI_API_KEY or ANTHROPIC_API_KEY in your environment first."
        )


## Concept 1 - The Anatomy of a Good Prompt

A reliable prompt usually spells out: **role/context, task, input data, constraints/format,** and optionally **examples**.
The cell below builds a prompt out of those pieces explicitly, then sends it.

In [ ]:
system_prompt = "You are a copywriter for an e-commerce team."
user_prompt = (
    "Task: Write a product description.\n"
    "Input: Stainless-steel insulated water bottle, 24oz, keeps drinks cold 24 hours.\n"
    "Constraints: exactly 3 sentences, friendly and upbeat tone, under 40 words total.\n"
)

print(ask(system_prompt, user_prompt))


## Concept 2 - Clarity & Specificity

Same task, two prompts. The first is vague; the second names the audience, format, length, and tone.
Run both and compare how much guessing the model has to do in each case.

In [ ]:
vague = "Write something about our product."
specific = (
    "Write a 3-sentence product description for a stainless-steel water bottle, "
    "for an e-commerce listing, in a friendly upbeat tone, under 40 words."
)

for label, prompt in [("VAGUE", vague), ("SPECIFIC", specific)]:
    print(f"--- {label} ---")
    print(ask("You are a product copywriter.", prompt))
    print()


## Concept 3 - System vs. User Prompts

The system prompt sets standing rules that persist for the whole conversation; the user prompt is the
specific ask for this turn. Below, the *user* question never changes - only the *system* prompt does.

In [ ]:
user_prompt = "Explain what a variable is."

systems = {
    "Neutral": "You are a helpful assistant.",
    "Kid-friendly tutor": "You are a patient tutor explaining concepts to a 10-year-old using simple, everyday analogies.",
    "Terse senior engineer": "You are a senior engineer. Be extremely concise and technical. No analogies, no small talk.",
}

for label, sys_prompt in systems.items():
    print(f"--- {label} ---")
    print(ask(sys_prompt, user_prompt))
    print()


## Concept 4 - Order & Structure

Delimiters (like `### HEADERS` or triple quotes) separate instructions from the data being processed,
which matters a lot once prompts get longer than a sentence or two.

In [ ]:
customer_review = (
    "I ordered the medium size but it runs small. The material feels premium "
    "though, and shipping was faster than expected. Customer service was slow "
    "to respond to my sizing question."
)

structured_prompt = f"""### INSTRUCTIONS
Summarize the review below in exactly 2 bullet points: one about the product, one about the service experience.

### TEXT
\"\"\"
{customer_review}
\"\"\"
"""

print(ask("You are a customer feedback analyst.", structured_prompt))


## Concept 5 - Workshop: One Prompt, Three Drafts

This mirrors the slide deck's live comparison. The same review is summarized three times, with each
version adding a bit more structure and constraint. Watch how the output changes.

In [ ]:
review = (
    "The onboarding flow was confusing at first, but once I found the settings "
    "page everything clicked. Support answered my email within an hour."
)

v1 = "Summarize this.\n\n" + review
v2 = "Summarize this customer review in 2 sentences.\n\n" + review
v3 = (
    "You are a support analyst. Summarize the review below in 2 sentences, "
    "noting the customer's main complaint (if any) and their overall sentiment "
    "(positive/neutral/negative).\n\n" + review
)

for label, prompt in [("V1 - bare", v1), ("V2 - some constraint", v2), ("V3 - role + constraint", v3)]:
    print(f"--- {label} ---")
    print(ask("You are a helpful assistant.", prompt))
    print()


## Challenges

Work through these on your own. No solutions are provided - write your own prompts and code below each prompt.


### Challenge 1 - Diagnose a Draft

Below are three real weak prompts. For each one, print (or write in a comment) at least one specific thing
missing from it: audience, format, length, tone, or a clearly stated task.

**Acceptance criteria:** your notebook identifies at least one concrete issue for each of the three prompts.

In [ ]:
weak_prompts = [
    "Tell me about our return policy.",
    "Make this better.",
    "Write an email.",
]

# TODO: for each prompt above, print what's missing and why it would confuse the model.


### Challenge 2 - Add Constraints

Take the vague prompt `"Write something about our product."` from Concept 2 and write your **own** improved
version (different wording than the one shown in class). Run it with `ask()` and print the result.

**Acceptance criteria:** your improved prompt specifies at least three of: audience, format, length, tone.

In [ ]:
# TODO: write your own specific version of the prompt and call ask(...) with it.
my_specific_prompt = ""


### Challenge 3 - Split the Stack

The prompt below stacks five unrelated jobs into one instruction. Split it into two (or more) clean,
single-task prompts, run each with `ask()`, and confirm each response does exactly one job.

**Acceptance criteria:** at least two separate prompts, each producing a focused, single-purpose response.

In [ ]:
overloaded_prompt = (
    "Summarize this article, translate the summary to Spanish, suggest a title, "
    "list three related topics, and rate how interesting it is out of 10: "
    "'Remote work has reshaped how teams collaborate, with async communication "
    "tools becoming central to daily operations.'"
)

# TODO: split overloaded_prompt into separate single-task prompts and run each one.


### Challenge 4 - System vs. User

Write your own system prompt persona (not one of the three shown in Concept 3) and test it against
**three different** user questions of your choice.

**Acceptance criteria:** one system prompt, three different user prompts, six total outputs printed.

In [ ]:
# TODO: define your own system prompt and three user questions, then call ask() for each combination.
my_system_prompt = ""
my_questions = []


### Challenge 5 - Write Your Own A-

Pick a real prompt you'd actually want to use (any topic). Write a "before" version and an "after" version,
run both, and add a markdown cell below explaining what specifically changed and why it helped.

**Acceptance criteria:** a before prompt, an after prompt, both outputs, and a short written explanation.

In [ ]:
# TODO: write your own before/after prompt pair and run both with ask().
before_prompt = ""
after_prompt = ""
